### Домашнее задание: анализ сетевого трафика с использованием Polars

#### 1. Загрузка и первичный анализ:

In [97]:
# Загрузите файл NF-CSE-CIC-IDS2018-V2.parquet в Polars DataFrame.
import polars as pl
df = pl.read_parquet("NF-CSE-CIC-IDS2018-V2.parquet")
display(df.head())


L4_SRC_PORT,L4_DST_PORT,PROTOCOL,L7_PROTO,IN_BYTES,IN_PKTS,OUT_BYTES,OUT_PKTS,TCP_FLAGS,CLIENT_TCP_FLAGS,SERVER_TCP_FLAGS,FLOW_DURATION_MILLISECONDS,DURATION_IN,DURATION_OUT,MIN_TTL,MAX_TTL,LONGEST_FLOW_PKT,SHORTEST_FLOW_PKT,MIN_IP_PKT_LEN,MAX_IP_PKT_LEN,SRC_TO_DST_SECOND_BYTES,DST_TO_SRC_SECOND_BYTES,RETRANSMITTED_IN_BYTES,RETRANSMITTED_IN_PKTS,RETRANSMITTED_OUT_BYTES,RETRANSMITTED_OUT_PKTS,SRC_TO_DST_AVG_THROUGHPUT,DST_TO_SRC_AVG_THROUGHPUT,NUM_PKTS_UP_TO_128_BYTES,NUM_PKTS_128_TO_256_BYTES,NUM_PKTS_256_TO_512_BYTES,NUM_PKTS_512_TO_1024_BYTES,NUM_PKTS_1024_TO_1514_BYTES,TCP_WIN_MAX_IN,TCP_WIN_MAX_OUT,ICMP_TYPE,ICMP_IPV4_TYPE,DNS_QUERY_ID,DNS_QUERY_TYPE,DNS_TTL_ANSWER,FTP_COMMAND_RET_CODE,Label,Attack
i32,i32,i8,f32,i32,i32,i32,i32,i16,i16,i16,i32,i32,i32,i16,i16,i32,i16,i16,i32,f64,f64,i32,i16,i32,i16,i64,i64,i32,i16,i16,i16,i32,i32,i32,i32,i16,i32,i16,i32,i8,i8,str
40894,22,6,92.0,3164,23,3765,21,27,27,27,0,0,0,63,63,1028,52,52,1028,3164.0,3765.0,0,0,0,0,25312000,30120000,33,7,1,2,1,26883,26847,0,0,0,0,0,0,1,"""SSH-Bruteforce"""
29622,3389,6,0.0,1919,14,2031,11,223,219,30,0,0,0,101,101,1195,40,40,1195,1919.0,2031.0,0,0,0,0,15352000,16248000,17,6,0,1,1,8192,64000,0,0,0,0,0,0,0,"""Benign"""
65456,53,17,0.0,116,2,148,2,0,0,0,0,0,0,128,128,74,58,58,74,116.0,148.0,0,0,0,0,928000,1184000,4,0,0,0,0,0,0,0,0,2511,1,5,0,0,"""Benign"""
57918,53,17,0.0,70,1,130,1,0,0,0,0,0,0,0,0,130,70,70,130,70.0,130.0,0,0,0,0,560000,1040000,1,1,0,0,0,0,0,0,0,3371,1,60,0,0,"""Benign"""
63269,80,6,7.0,232,5,1136,4,223,222,27,4294827,140,0,127,127,1004,40,40,1004,232.0,1136.0,0,0,0,0,8000,9088000,8,0,0,1,0,8192,26883,0,0,0,0,0,0,1,"""DDoS attacks-LOIC-HTTP"""


In [98]:
# Выведите
# форму датасета (количество строк и колонок);
print(f"Количество строк и колонок датасета: \n строк: {df.shape[0]}, колонок: {df.shape[1]}")
# типы колонок (последние пять);
print(f"Типы последних пяти колонок: {df.dtypes[-5:]}")
# уникальные значения в колонке Label;
unique_label_lazy = (
    df.lazy()
    .select(
        pl.col("Label").unique(maintain_order=True).alias("unique_label_cat"),
        pl.col("Label").unique_counts().alias("unique_label_counts")
    )
    .collect()
)
print(f"Уникальные значения в колонке Label: \n {unique_label_lazy}")
# уникальные значения в колонке Attack.
unique_Attack_lazy = (
    df.lazy()
    .select(
        pl.col("Attack").unique(maintain_order=True).alias("unique_Attack_cat"),
        pl.col("Attack").unique_counts().alias("unique_Attack_counts")
    )
    .sort("unique_Attack_counts",descending=True)
    .collect()
)
print(f"Уникальные значения в колонке Attack: \n {unique_Attack_lazy}")


Количество строк и колонок датасета: 
 строк: 17129715, колонок: 43
Типы последних пяти колонок: [Int16, Int32, Int8, Int8, String]
Уникальные значения в колонке Label: 
 shape: (2, 2)
┌──────────────────┬─────────────────────┐
│ unique_label_cat ┆ unique_label_counts │
│ ---              ┆ ---                 │
│ i8               ┆ u32                 │
╞══════════════════╪═════════════════════╡
│ 1                ┆ 2028030             │
│ 0                ┆ 15101685            │
└──────────────────┴─────────────────────┘
Уникальные значения в колонке Attack: 
 shape: (15, 2)
┌────────────────────────┬──────────────────────┐
│ unique_Attack_cat      ┆ unique_Attack_counts │
│ ---                    ┆ ---                  │
│ str                    ┆ u32                  │
╞════════════════════════╪══════════════════════╡
│ Benign                 ┆ 15101685             │
│ DDOS attack-HOIC       ┆ 1066881              │
│ DoS attacks-Hulk       ┆ 432648               │
│ DDoS attacks-L

##### Выводы:
- Наибольшее количество атак составляют атаки категрии "DDOS attack-HOIC";
- Наименьшее количество атак составляют атаки категрии "SQL Injection";
- Большинство наблюдений не являются атаками.

#### 2. Распределение по меткам:

In [99]:
# Подсчитайте:
# сколько записей помечено как Label == 0 ("Benign");
Benign_label_lazy = (
    df.lazy()
    .filter(pl.col("Label") == 0)
    .select(pl.len())
    .collect()
)
print(f"Записей помеченных как Label == 0 (Benign):{Benign_label_lazy['len'][0]}")

# сколько записей помечено как Label == 1 ("Attack").
Attack_label_lazy = (
    df.lazy()
    .filter(pl.col("Label") == 1)
    .select(pl.len())
    .collect()
)

print(f"Записей помеченных как Label == 1 (Attack):{Attack_label_lazy['len'][0]}")

# Выведите процентное соотношение.
p_ones = (df["Label"] == 0).mean() * 100
p_zeros = (df["Label"] == 1).mean() * 100
print(f"Процентное соотношение: Benign: {p_ones:.2f}% и Attack: {p_zeros:.2f}%")




Записей помеченных как Label == 0 (Benign):15101685
Записей помеченных как Label == 1 (Attack):2028030
Процентное соотношение: Benign: 88.16% и Attack: 11.84%


##### Выводы:
- Атак зафиксировано не более 12% среди всего сетевого трафика.
- Можно сделать вывод о качественной системе безопасности.
 

#### 3. Создание бинарного признака:

In [100]:
# Добавьте колонку is_attack, которая:
# 1, если Label != 0 (атака);
# 0, если Label == 0 (норма).

df = df.with_columns(
    pl.when(pl.col('Label') != 0 )
    .then(pl.lit(1))
    .when(pl.col('Label') == 0 )
    .then(pl.lit(0))
    .alias('is_attack')
)

print(f"Добавлена колонка is_attack: \n {df.head()}")




Добавлена колонка is_attack: 
 shape: (5, 44)
┌─────────────┬────────────┬──────────┬──────────┬───┬────────────┬───────┬────────────┬───────────┐
│ L4_SRC_PORT ┆ L4_DST_POR ┆ PROTOCOL ┆ L7_PROTO ┆ … ┆ FTP_COMMAN ┆ Label ┆ Attack     ┆ is_attack │
│ ---         ┆ T          ┆ ---      ┆ ---      ┆   ┆ D_RET_CODE ┆ ---   ┆ ---        ┆ ---       │
│ i32         ┆ ---        ┆ i8       ┆ f32      ┆   ┆ ---        ┆ i8    ┆ str        ┆ i32       │
│             ┆ i32        ┆          ┆          ┆   ┆ i8         ┆       ┆            ┆           │
╞═════════════╪════════════╪══════════╪══════════╪═══╪════════════╪═══════╪════════════╪═══════════╡
│ 40894       ┆ 22         ┆ 6        ┆ 92.0     ┆ … ┆ 0          ┆ 1     ┆ SSH-Brutef ┆ 1         │
│             ┆            ┆          ┆          ┆   ┆            ┆       ┆ orce       ┆           │
│ 29622       ┆ 3389       ┆ 6        ┆ 0.0      ┆ … ┆ 0          ┆ 0     ┆ Benign     ┆ 0         │
│ 65456       ┆ 53         ┆ 17       ┆ 0.0  

#### 4. Агрегация по типам атак:

In [101]:
# Отфильтруйте только атаки (Label != 0).
# Сгруппируйте данные по колонке Attack.
# Посчитайте:
# среднюю длительность потока (FLOW_DURATION_MILLISECONDS);
# среднее количество входящих байт (IN_BYTES);
# общее количество записей для каждого типа атаки.
# Отсортируйте по убыванию avg_in_bytes.

result_Attack = (
    df.lazy()
    .filter(pl.col("Label") != 0)
    .group_by('Attack')
    .agg([
    pl.col("FLOW_DURATION_MILLISECONDS").mean().alias("flow_duration"),
    pl.col("IN_BYTES").mean().alias("avg_in_bytes"),
    pl.len().alias("count_attack"),
    ])
    .sort("avg_in_bytes", descending=True)
    .collect()
)
print(f"Аргрегация по типам атак: \n {result_Attack}")


# Сохраните итоговую агрегацию (по типам атак) в файл attack_summary_by_type.parquet.
# result_Attack.write_parquet("attack_summary_by_type.parquet")


Аргрегация по типам атак: 
 shape: (14, 4)
┌──────────────────────────┬───────────────┬──────────────┬──────────────┐
│ Attack                   ┆ flow_duration ┆ avg_in_bytes ┆ count_attack │
│ ---                      ┆ ---           ┆ ---          ┆ ---          │
│ str                      ┆ f64           ┆ f64          ┆ u32          │
╞══════════════════════════╪═══════════════╪══════════════╪══════════════╡
│ DDOS attack-LOIC-UDP     ┆ 4.1959e6      ┆ 5.8540e6     ┆ 2112         │
│ DDoS attacks-LOIC-HTTP   ┆ 3.7241e6      ┆ 25991.904925 ┆ 207078       │
│ Brute Force -XSS         ┆ 3.7568e6      ┆ 16871.281553 ┆ 927          │
│ Brute Force -Web         ┆ 3.7553e6      ┆ 9797.653756  ┆ 2143         │
│ SSH-Bruteforce           ┆ 733291.768981 ┆ 5828.140747  ┆ 94979        │
│ …                        ┆ …             ┆ …            ┆ …            │
│ Infilteration            ┆ 234910.506514 ┆ 791.858613   ┆ 115513       │
│ Bot                      ┆ 1.2340e6      ┆ 677.354225  

##### Выводы:
- Атаки категории "DDOS attack-LOIC-UDP" имеют наибольшее количество входящих байт, но количество зафиксированных атак не значительно 2112;
- Атаки категории "DoS attacks-SlowHTTPTest" имеют наименьшее количество входящих байт.

#### 5. Топ-3 атак по трафику:

In [102]:
# Выведите топ-3 типа атак по среднему объёму входящего трафика (avg_in_bytes).
result_top_3_avg_in_bytes = (
    result_Attack.lazy()
    .sort("avg_in_bytes", descending=True)
    .head(3)
    .collect()
)

print(f"Топ-3 типа атак по среднему объёму входящего трафика: \n {result_top_3_avg_in_bytes}")

Топ-3 типа атак по среднему объёму входящего трафика: 
 shape: (3, 4)
┌────────────────────────┬───────────────┬──────────────┬──────────────┐
│ Attack                 ┆ flow_duration ┆ avg_in_bytes ┆ count_attack │
│ ---                    ┆ ---           ┆ ---          ┆ ---          │
│ str                    ┆ f64           ┆ f64          ┆ u32          │
╞════════════════════════╪═══════════════╪══════════════╪══════════════╡
│ DDOS attack-LOIC-UDP   ┆ 4.1959e6      ┆ 5.8540e6     ┆ 2112         │
│ DDoS attacks-LOIC-HTTP ┆ 3.7241e6      ┆ 25991.904925 ┆ 207078       │
│ Brute Force -XSS       ┆ 3.7568e6      ┆ 16871.281553 ┆ 927          │
└────────────────────────┴───────────────┴──────────────┴──────────────┘


#### 6. Распределение по протоколам:

In [103]:
# Выведите:
# общее распределение по PROTOCOL;
result_full_protocol = (
    df.lazy()
    .group_by("PROTOCOL")
    .agg([
        pl.len().alias("count"),
        (pl.len() / len(df) * 100).round(2).alias("percent") 
    ])
    .sort("count", descending=True)
    .collect()
)
print(f"Общее распределение по столбцу PROTOCOL: \n {result_full_protocol}")

# распределение по PROTOCOL только для Benign;
result_Benign_protocol = (
    df.lazy()
    .filter(pl.col("is_attack") == 0)
    .group_by("PROTOCOL")
    .agg([
        pl.len().alias("count"),
        (pl.len() / len(df) * 100).round(2).alias("percent") 
    ])
    .sort("count", descending=True)
    .collect()
)
print(f"Распределение по PROTOCOL только для Benign: \n {result_Benign_protocol}")


# распределение по PROTOCOL только для Attack (с разбивкой по Attack);
result_Attack_protocol = (
    df.lazy()
    .filter(pl.col("is_attack") == 1) 
    .group_by(["PROTOCOL", "Attack"])
    .agg(pl.len().alias("count"))
    .sort(["PROTOCOL", "count"], descending=[False, True])
    .collect()
)

print(f"Распределение по PROTOCOL только для Attack (с разбивкой по Attack): \n {result_Attack_protocol}")

# сравнение PROTOCOL между Benign и Attack.
# Шаг 1 агрегация  ('Benign')
benign_grouped = (
    df.lazy()
    .filter(pl.col('is_attack') == 0)
    .group_by('PROTOCOL')
    .agg(pl.len().alias('Benign'))
)

# Шаг 2 агрегация  ('Attack')
attack_grouped = (
    df.lazy()
    .filter(pl.col('is_attack') == 1)
    .group_by('PROTOCOL')
    .agg(pl.len().alias('Attack'))
)

# Шаг 3 Объединяем 
final_result = benign_grouped.join(attack_grouped, on='PROTOCOL', how='full').fill_null(0)

# Собираем и сортируем результат
result_protocol_Benign_Attack =  (
    final_result.lazy()    
    .sort(["PROTOCOL", "Benign","Attack"], descending=False)
    .collect()
)

print(f"Cравнение PROTOCOL между Benign и Attack: \n {result_protocol_Benign_Attack}")




Общее распределение по столбцу PROTOCOL: 
 shape: (6, 3)
┌──────────┬─────────┬─────────┐
│ PROTOCOL ┆ count   ┆ percent │
│ ---      ┆ ---     ┆ ---     │
│ i8       ┆ u32     ┆ f64     │
╞══════════╪═════════╪═════════╡
│ 6        ┆ 9346287 ┆ 54.56   │
│ 17       ┆ 7776756 ┆ 45.4    │
│ 1        ┆ 4857    ┆ 0.03    │
│ 2        ┆ 976     ┆ 0.01    │
│ 58       ┆ 836     ┆ 0.0     │
│ 47       ┆ 3       ┆ 0.0     │
└──────────┴─────────┴─────────┘
Распределение по PROTOCOL только для Benign: 
 shape: (6, 3)
┌──────────┬─────────┬─────────┐
│ PROTOCOL ┆ count   ┆ percent │
│ ---      ┆ ---     ┆ ---     │
│ i8       ┆ u32     ┆ f64     │
╞══════════╪═════════╪═════════╡
│ 17       ┆ 7688529 ┆ 44.88   │
│ 6        ┆ 7406897 ┆ 43.24   │
│ 1        ┆ 4537    ┆ 0.03    │
│ 2        ┆ 883     ┆ 0.01    │
│ 58       ┆ 836     ┆ 0.0     │
│ 47       ┆ 3       ┆ 0.0     │
└──────────┴─────────┴─────────┘
Распределение по PROTOCOL только для Attack (с разбивкой по Attack): 
 shape: (20, 3)
┌───

##### Выводы:
- Самым активными протокалами являются протоколы 6  и  17, на них приходится 99 % трафика.
- Наибольшее количество атак также приходятся на протоколы 6 и 17 и самая в них распространённый тип атак это DDOS attack.
- атаки по 47 т 58 протоколо в сетевом трафике отстутствуют. <br>
Необходимо сосредоточиться на защите имено 6 и 17 протоколов.

#### 7. Сравнение метрик: Benign vs Attack:

In [104]:
# Сгруппируйте по is_attack и посчитайте:
# средние значения IN_BYTES, OUT_BYTES, FLOW_DURATION_MILLISECONDS;
# общее количество записей.


result_is_attack = (
    df.lazy()
    .group_by('is_attack')
    .agg([
    pl.col("IN_BYTES").mean().alias("avg_in_bytes"),
    pl.col("OUT_BYTES").mean().alias("avg_out_bytes"),
    pl.col("FLOW_DURATION_MILLISECONDS").mean().alias("avg_flow_duration"),
    pl.len().alias("count_is_attack"),
    ])
    .collect()
)
print(f"Группировка по столбцу is_attack согласно заданию: \n {result_is_attack}")




Группировка по столбцу is_attack согласно заданию: 
 shape: (2, 5)
┌───────────┬──────────────┬───────────────┬───────────────────┬─────────────────┐
│ is_attack ┆ avg_in_bytes ┆ avg_out_bytes ┆ avg_flow_duration ┆ count_is_attack │
│ ---       ┆ ---          ┆ ---           ┆ ---               ┆ ---             │
│ i32       ┆ f64          ┆ f64           ┆ f64               ┆ u32             │
╞═══════════╪══════════════╪═══════════════╪═══════════════════╪═════════════════╡
│ 1         ┆ 10013.788006 ┆ 2301.412192   ┆ 3.7472e6          ┆ 2028030         │
│ 0         ┆ 867.640543   ┆ 8253.537161   ┆ 185715.810107     ┆ 15101685        │
└───────────┴──────────────┴───────────────┴───────────────────┴─────────────────┘


#### 8. (Бонус) Эвристика детектирования:

In [105]:
# Создайте колонку is_suspicious, которая равна:
# 1, если:
# bytes_ratio > 10 (входящий трафик >> исходящего);
# FLOW_DURATION_MILLISECONDS < 500 (короткий поток);
# IN_PKTS > 10 (много пакетов).
# 0 — в противном случае.
# Подсчитайте:
# сколько записей помечено как is_suspicious == 1;
# сколько из них на самом деле являются атаками (Label != 0).
# Вычислите точность эвристики: true_attacks / total_flagged.

df = df.with_columns(
    pl.when(((pl.col('IN_BYTES') / (pl.col('OUT_BYTES')+1)) > 10 ) & (pl.col('FLOW_DURATION_MILLISECONDS') < 500) & (pl.col('IN_PKTS')  > 10))
    .then(pl.lit(1))
    .otherwise(pl.lit(0))
    .cast(pl.Int8) 
    .alias('is_suspicious')
)

print(f"Создана колонка is_suspicious согласно усовиям ТЗ: \n {df.head()}")

# сколько записей помечено как is_suspicious == 1;


is_suspicious_lazy = (
    df.lazy()
    .filter(pl.col("is_suspicious") == 1)
    .select(pl.len())
    .collect()
)
print(f"Количество записей помеченых, как is_suspicious == 1: {is_suspicious_lazy['len'][0]}")


# сколько из них на самом деле являются атаками (Label != 0).
is_suspicious_attack_lazy = (
    df.lazy()
    .filter((pl.col("is_suspicious") == 1) & (pl.col("Label") != 0 ))
    .select(pl.len())
    .collect()
)
print(f"Количество записей помеченых, как is_suspicious == 1 и являющимися атаками Label != 0): {is_suspicious_attack_lazy['len'][0]}")

# Вычислите точность эвристики: true_attacks / total_flagged.
is_suspicious_itog = ((is_suspicious_attack_lazy['len'] / is_suspicious_lazy['len'])*100).round(2).cast(pl.String)
print(f"Точность эвристики составляет: {is_suspicious_itog[0]}%")




Создана колонка is_suspicious согласно усовиям ТЗ: 
 shape: (5, 45)
┌─────────────┬────────────┬──────────┬──────────┬───┬───────┬────────────┬───────────┬────────────┐
│ L4_SRC_PORT ┆ L4_DST_POR ┆ PROTOCOL ┆ L7_PROTO ┆ … ┆ Label ┆ Attack     ┆ is_attack ┆ is_suspici │
│ ---         ┆ T          ┆ ---      ┆ ---      ┆   ┆ ---   ┆ ---        ┆ ---       ┆ ous        │
│ i32         ┆ ---        ┆ i8       ┆ f32      ┆   ┆ i8    ┆ str        ┆ i32       ┆ ---        │
│             ┆ i32        ┆          ┆          ┆   ┆       ┆            ┆           ┆ i8         │
╞═════════════╪════════════╪══════════╪══════════╪═══╪═══════╪════════════╪═══════════╪════════════╡
│ 40894       ┆ 22         ┆ 6        ┆ 92.0     ┆ … ┆ 1     ┆ SSH-Brutef ┆ 1         ┆ 0          │
│             ┆            ┆          ┆          ┆   ┆       ┆ orce       ┆           ┆            │
│ 29622       ┆ 3389       ┆ 6        ┆ 0.0      ┆ … ┆ 0     ┆ Benign     ┆ 0         ┆ 0          │
│ 65456       ┆ 53     

##### Выводы:
- Эвристика детектирования согласно текущим условиям точна только на 80%. 
- Считаю необходимо повысить точность проанализировав все возможные сочетания атак и скорректировать условия детектирования для достижения более выской точности. 
- 20% ошибок детектирования слишком большой показатель и его необходимо уменьшать, увеличив точность детектирования.
